# Подготовка данных к анализу

## Цель


- Переименовать столбцы в стиле `snake_case` для упрощения чтения кода;
- Преобразовать `Timely response?` в логический признак.

Пропуски в необязательных полях сохраняются. Строки не удаляются
без обоснованной причины.



In [22]:
from pathlib import Path
import pandas as pd

In [23]:
CURRENT_DIR=Path.cwd()
PROJECT_ROOT=(
    CURRENT_DIR.parent
    if CURRENT_DIR.name=="notebooks"
    else CURRENT_DIR
)
DATA_PATH = (PROJECT_ROOT /"data"/"processed"/"complaints_initialized.pkl")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Файл не найден: {DATA_PATH}")
clean_df=pd.read_pickle(DATA_PATH)
print("Файл:", DATA_PATH)
print("Размер таблицы:", clean_df.shape) #ожидаемый результат (141822, 19)

Файл: /Users/zemfiraramazanova/PycharmProjects/financial-complaints-radar/data/processed/complaints_initialized.pkl
Размер таблицы: (141822, 19)


#### Переименуем столбцы

Названия приводятся к формату `snake_case`, чтобы их было
удобно использовать

In [24]:
column_names={
    "Date received": "date_received",
    "Product": "product",
    "Sub-product": "sub_product",
    "Issue": "issue",
    "Sub-issue": "sub_issue",
    "Company public response": "company_public_response",
    "Company": "company",
    "State": "state",
    "ZIP code": "zip_code",
    "Tags": "tags",
    "Submitted via": "submitted_via",
    "Date sent to company": "date_sent_to_company",
    "Company response to consumer": "company_response",
    "Timely response?": "timely_response",
    "Complaint ID": "complaint_id",
    "source": "source_file"
}
clean_df=clean_df.rename(
    columns=column_names
)
clean_df.columns.tolist()

['date_received',
 'product',
 'sub_product',
 'issue',
 'sub_issue',
 'company_public_response',
 'company',
 'state',
 'zip_code',
 'tags',
 'submitted_via',
 'date_sent_to_company',
 'company_response',
 'timely_response',
 'complaint_id',
 'source_file',
 'year',
 'month',
 'year_month']

#### Своевременность ответа

Исходный столбец `timely_response` содержит значения `Yes` и `No`.
Для расчётов создаётся логический признак `is_timely`, но сохраним оригинальный столбец для контроля

In [25]:
timely_mapping = {
    "Yes": True,
    "No": False,
}
clean_df["is_timely"] = (clean_df["timely_response"]
    .map(timely_mapping)
    .astype("boolean")
)

#### Время передачи жалобы

Создаётся признак `days_to_send` — количество дней между
получением жалобы CFPB и её передачей компании.


In [26]:
clean_df["days_to_send"]=(clean_df["date_sent_to_company"]-clean_df["date_received"]).dt.total_seconds()/86_400

#### Проверка

In [27]:
cleaning_report=pd.Series(
    {
        "Количество строк": len(clean_df),
        "Пропущено complaint_id": (clean_df["complaint_id"].isna().sum()),
        "Дубликатов complaint_id": (clean_df["complaint_id"].duplicated().sum()),
        "Пропущено is_timely": (clean_df["is_timely"].isna().sum()),
        "Отрицательных days_to_send": (clean_df["days_to_send"].lt(0).sum()),
        "Пропущено issue": (clean_df["issue"].isna().sum())
    },
    name="value",
)

cleaning_report

Количество строк              141822
Пропущено complaint_id             0
Дубликатов complaint_id            0
Пропущено is_timely                0
Отрицательных days_to_send         0
Пропущено issue                    1
Name: value, dtype: int64

#### Сохраним отчищенные данные

In [28]:
OUTPUT_PATH = (PROJECT_ROOT/"data"/"processed"/"complaints_clean.pkl")
clean_df.to_pickle(OUTPUT_PATH)
print("Файл сохранён:", OUTPUT_PATH)
print("Файл существует:", OUTPUT_PATH.exists())

Файл сохранён: /Users/zemfiraramazanova/PycharmProjects/financial-complaints-radar/data/processed/complaints_clean.pkl
Файл существует: True


## Итоги отчистки данных

- Названия столбцов приведены к формату `snake_case`
-
- `is_timely` — признак своевременного ответа;
- `days_to_send`-время передачи жалобы компании в днях.

Чистый набор сохранён в `data/processed/complaints_clean.pkl` и готов к анализу